In [0]:
# ============================================================
# DATA CATALOG — Etapa 1: inventário das tabelas principais
# ============================================================

CATALOG = "mvp_engenharia_de_dados_puc_rio"
SCHEMA = "dados_tse"

tabelas_catalogo = [
    "silver_candidatos_deputado_estadual_sp",
    "gold_comparacao_candidatos_2022_2026",
    "gold_distribuicao_raca",
    "gold_distribuicao_genero"
]

print("========== INVENTÁRIO DO DATA CATALOG ==========")

for tabela in tabelas_catalogo:
    nome_completo = f"{CATALOG}.{SCHEMA}.{tabela}"
    df = spark.table(nome_completo)

    print(f"\n{'=' * 70}")
    print(f"TABELA: {tabela}")
    print(f"Registros: {df.count()}")
    print(f"Colunas: {len(df.columns)}")
    print("=" * 70)

    for campo in df.schema.fields:
        print(
            f"{campo.name:<40} "
            f"{campo.dataType.simpleString()}"
        )

In [0]:
# ============================================================
# DATA CATALOG — Etapa 2:
# documentar tabela Silver no Unity Catalog
# ============================================================

TABELA_SILVER = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "silver_candidatos_deputado_estadual_sp"
)

descricao_tabela_silver = """
Tabela Silver consolidada de candidaturas ao cargo de Deputado Estadual
no estado de São Paulo, para as eleições de 2022 e 2026.

Origem:
- bronze_consulta_cand_2022
- bronze_consulta_cand_2026
- bronze_consulta_cand_complementar_2022
- bronze_consulta_cand_complementar_2026

Granularidade:
uma linha por candidatura e ano eleitoral, identificada logicamente por
ano_eleicao + sq_candidato.

Principais tratamentos:
- filtro para Deputado Estadual em São Paulo;
- padronização dos tipos entre 2022 e 2026;
- normalização da data da eleição;
- identificação de CPF válido para cruzamento;
- enriquecimento com dados da fonte complementar;
- preservação das categorias e situações originais do TSE.
"""

spark.sql(f"""
COMMENT ON TABLE {TABELA_SILVER}
IS '{descricao_tabela_silver.strip().replace("'", "''")}'
""")

print("Comentário da tabela Silver aplicado com sucesso.")

In [0]:
# ============================================================
# DATA CATALOG — Etapa 3:
# comentários das colunas da Silver
# ============================================================

comentarios_silver = {

    "ano_eleicao":
        "Ano da eleição. Origem: ANO_ELEICAO das fontes CONSULTA_CAND do TSE. "
        "Domínio analisado: 2022 e 2026.",

    "sq_candidato":
        "Identificador sequencial da candidatura atribuído pelo TSE. "
        "Origem: SQ_CANDIDATO. Integra a chave lógica ano_eleicao + sq_candidato.",

    "dt_eleicao_original":
        "Valor original de DT_ELEICAO convertido para string para preservar "
        "as diferenças de tipagem existentes entre as fontes de 2022 e 2026.",

    "data_eleicao":
        "Data da eleição padronizada como DATE. "
        "Valores observados: 2022-10-02 e 2026-10-04.",

    "uf":
        "Unidade da Federação da candidatura. Origem: SG_UF do TSE. "
        "Recorte deste MVP: SP.",

    "cd_cargo":
        "Código do cargo eleitoral segundo o TSE. "
        "Recorte utilizado no MVP: código 7, Deputado Estadual.",

    "cargo":
        "Descrição do cargo eleitoral. Origem: DS_CARGO. "
        "Nesta tabela: DEPUTADO ESTADUAL.",

    "nr_candidato":
        "Número do candidato utilizado na urna. Origem: NR_CANDIDATO.",

    "nome_candidato":
        "Nome completo do candidato conforme cadastro do TSE. "
        "Origem: NM_CANDIDATO.",

    "nome_urna_candidato":
        "Nome do candidato apresentado na urna. Origem: NM_URNA_CANDIDATO.",

    "cpf_candidato":
        "CPF disponibilizado na fonte TSE, utilizado apenas internamente para "
        "comparações entre eleições. Não deve ser exposto individualmente na documentação pública.",

    "cpf_valido_para_cruzamento":
        "Indicador técnico criado no pipeline. TRUE quando cpf_candidato > 0. "
        "Valores especiais como -4 não são utilizados no cruzamento entre eleições.",

    "cd_genero":
        "Código de gênero segundo classificação da fonte TSE. Origem: CD_GENERO.",

    "genero":
        "Descrição de gênero conforme a fonte TSE. Origem: DS_GENERO. "
        "Categorias observadas incluem MASCULINO, FEMININO e NÃO DIVULGÁVEL.",

    "cd_cor_raca":
        "Código de cor/raça segundo classificação da fonte TSE. Origem: CD_COR_RACA.",

    "cor_raca":
        "Descrição da cor/raça autodeclarada conforme fonte TSE. Origem: DS_COR_RACA. "
        "As categorias originais são preservadas sem agrupamentos adicionais.",

    "nr_partido":
        "Número do partido político da candidatura. Origem: NR_PARTIDO.",

    "sigla_partido":
        "Sigla do partido político da candidatura. Origem: SG_PARTIDO.",

    "nome_partido":
        "Nome completo do partido político. Origem: NM_PARTIDO.",

    "cd_situacao_candidatura":
        "Código da situação da candidatura conforme fonte TSE. "
        "Origem: CD_SITUACAO_CANDIDATURA.",

    "situacao_candidatura":
        "Descrição da situação da candidatura conforme TSE. "
        "Em 2022 foram observados APTO e INAPTO; em 2026 o valor disponível é #NE.",

    "cd_situacao_turno":
        "Código da situação total do candidato no turno. "
        "Origem: CD_SIT_TOT_TURNO.",

    "situacao_turno":
        "Descrição da situação total do candidato no turno. "
        "Origem: DS_SIT_TOT_TURNO.",

    "detalhe_situacao_candidatura":
        "Detalhamento da situação da candidatura proveniente da fonte "
        "CONSULTA_CAND_COMPLEMENTAR.",

    "nacionalidade":
        "Descrição da nacionalidade do candidato. "
        "Origem: DS_NACIONALIDADE da fonte complementar do TSE.",

    "municipio_nascimento":
        "Município de nascimento do candidato conforme fonte complementar do TSE. "
        "Origem: NM_MUNICIPIO_NASCIMENTO.",

    "idade_data_posse":
        "Idade calculada pelo TSE para a data da posse. "
        "Origem: NR_IDADE_DATA_POSSE.",

    "st_quilombola":
        "Indicador de informação quilombola conforme disponibilizado na fonte complementar do TSE. "
        "Domínio preservado da origem.",

    "etnia_indigena":
        "Descrição da etnia indígena conforme fonte complementar do TSE. "
        "Origem: DS_ETNIA_INDIGENA.",

    "limite_despesa_campanha":
        "Valor máximo de despesas de campanha informado na fonte complementar do TSE. "
        "Origem: VR_DESPESA_MAX_CAMPANHA.",

    "st_reeleicao":
        "Indicador de reeleição conforme disponibilizado na fonte complementar do TSE. "
        "Origem: ST_REELEICAO.",

    "st_declarar_bens":
        "Indicador relacionado à declaração de bens conforme fonte complementar do TSE. "
        "Origem: ST_DECLARAR_BENS."
}

for coluna, comentario in comentarios_silver.items():
    comentario_sql = comentario.replace("'", "''")

    spark.sql(f"""
        ALTER TABLE {TABELA_SILVER}
        ALTER COLUMN {coluna}
        COMMENT '{comentario_sql}'
    """)

print(
    f"Comentários aplicados: {len(comentarios_silver)} "
    "colunas da tabela Silver."
)

In [0]:
# ============================================================
# DATA CATALOG — Etapa 4:
# documentar tabelas Gold
# ============================================================

def aplicar_documentacao_tabela(nome_tabela, descricao_tabela, comentarios_colunas):
    descricao_sql = descricao_tabela.strip().replace("'", "''")

    spark.sql(f"""
        COMMENT ON TABLE {nome_tabela}
        IS '{descricao_sql}'
    """)

    for coluna, comentario in comentarios_colunas.items():
        comentario_sql = comentario.replace("'", "''")

        spark.sql(f"""
            ALTER TABLE {nome_tabela}
            ALTER COLUMN {coluna}
            COMMENT '{comentario_sql}'
        """)

    print(
        f"Documentação aplicada em {nome_tabela}: "
        f"{len(comentarios_colunas)} colunas."
    )


# ============================================================
# GOLD 1 — Comparação de candidatos 2022 x 2026
# ============================================================

TABELA_GOLD_COMPARACAO = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "gold_comparacao_candidatos_2022_2026"
)

descricao_gold_comparacao = """
Tabela Gold analítica para comparação de candidatos a Deputado Estadual
em São Paulo entre as eleições de 2022 e 2026.

Origem:
silver_candidatos_deputado_estadual_sp.

Granularidade:
uma linha por CPF válido distinto no universo combinado de 2022 e 2026.

Regra de identificação:
somente CPFs com cpf_valido_para_cruzamento = TRUE são utilizados.

Tratamento de duplicidades em 2022:
quando um mesmo CPF possuía mais de uma candidatura no mesmo ano,
foi priorizado o registro com situação APTO e detalhe DEFERIDO.

Objetivo:
responder às análises de recorrência de candidatos entre os anos e
identificar alteração de sigla partidária entre candidatos presentes
nas duas eleições.
"""

comentarios_gold_comparacao = {
    "cpf_candidato":
        "CPF utilizado internamente como identificador da pessoa entre eleições. "
        "Somente valores considerados válidos no pipeline são incluídos nesta Gold.",

    "nome_candidato":
        "Nome de referência do candidato. Utiliza prioritariamente o nome de 2026 "
        "quando disponível; caso contrário, utiliza o nome de 2022.",

    "participou_2022":
        "Indicador booleano que informa se o CPF possui candidatura no recorte de 2022.",

    "participou_2026":
        "Indicador booleano que informa se o CPF possui candidatura no recorte de 2026.",

    "situacao_comparacao":
        "Classificação analítica da participação entre os anos. "
        "Domínio: SOMENTE_2022, SOMENTE_2026 ou AMBOS_ANOS.",

    "nome_2022":
        "Nome do candidato na candidatura de 2022. Origem: Silver.",

    "sq_candidato_2022":
        "Identificador sequencial da candidatura selecionada para 2022.",

    "nr_candidato_2022":
        "Número de urna do candidato em 2022.",

    "partido_2022":
        "Sigla partidária da candidatura em 2022.",

    "genero_2022":
        "Descrição de gênero da candidatura em 2022, preservada da fonte TSE.",

    "cor_raca_2022":
        "Descrição de cor/raça da candidatura em 2022, preservada da fonte TSE.",

    "situacao_2022":
        "Situação da candidatura de 2022 conforme disponibilizada pelo TSE.",

    "nome_2026":
        "Nome do candidato na candidatura de 2026. Origem: Silver.",

    "sq_candidato_2026":
        "Identificador sequencial da candidatura selecionada para 2026.",

    "nr_candidato_2026":
        "Número de urna do candidato em 2026.",

    "partido_2026":
        "Sigla partidária da candidatura em 2026.",

    "genero_2026":
        "Descrição de gênero da candidatura em 2026, preservada da fonte TSE.",

    "cor_raca_2026":
        "Descrição de cor/raça da candidatura em 2026, preservada da fonte TSE.",

    "situacao_2026":
        "Situação da candidatura de 2026 conforme disponibilizada pelo TSE.",

    "mudou_partido":
        "Indicador calculado somente para candidatos presentes nos dois anos. "
        "TRUE quando partido_2022 é diferente de partido_2026; FALSE quando é igual. "
        "Fica nulo quando o candidato não participa dos dois anos."
}

aplicar_documentacao_tabela(
    TABELA_GOLD_COMPARACAO,
    descricao_gold_comparacao,
    comentarios_gold_comparacao
)


# ============================================================
# GOLD 2 — Distribuição por cor/raça
# ============================================================

TABELA_GOLD_RACA = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "gold_distribuicao_raca"
)

descricao_gold_raca = """
Tabela Gold agregada para comparação da distribuição de cor/raça das
candidaturas a Deputado Estadual em São Paulo entre 2022 e 2026.

Origem:
silver_candidatos_deputado_estadual_sp.

Granularidade:
uma linha por categoria de cor/raça observada no universo combinado.

As categorias originais do TSE são preservadas.
A tabela contém quantidade, participação percentual por ano,
variação percentual da quantidade e variação em pontos percentuais.
"""

comentarios_gold_raca = {
    "cor_raca":
        "Categoria de cor/raça conforme classificação original do TSE.",

    "qtd_2022":
        "Quantidade de registros de candidaturas da categoria em 2022.",

    "pct_2022":
        "Participação percentual da categoria sobre o total de candidaturas de 2022.",

    "qtd_2026":
        "Quantidade de registros de candidaturas da categoria em 2026.",

    "pct_2026":
        "Participação percentual da categoria sobre o total de candidaturas de 2026.",

    "variacao_qtd_pct":
        "Variação percentual da quantidade entre 2022 e 2026, calculada como "
        "(qtd_2026 - qtd_2022) / qtd_2022 * 100.",

    "variacao_pontos_percentuais":
        "Diferença entre pct_2026 e pct_2022, expressa em pontos percentuais."
}

aplicar_documentacao_tabela(
    TABELA_GOLD_RACA,
    descricao_gold_raca,
    comentarios_gold_raca
)


# ============================================================
# GOLD 3 — Distribuição por gênero
# ============================================================

TABELA_GOLD_GENERO = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "gold_distribuicao_genero"
)

descricao_gold_genero = """
Tabela Gold agregada para comparação da distribuição de gênero das
candidaturas a Deputado Estadual em São Paulo entre 2022 e 2026.

Origem:
silver_candidatos_deputado_estadual_sp.

Granularidade:
uma linha por categoria de gênero observada no universo combinado.

As categorias originais do TSE são preservadas.
A tabela contém quantidade, participação percentual por ano,
variação percentual da quantidade e variação em pontos percentuais.
"""

comentarios_gold_genero = {
    "genero":
        "Categoria de gênero conforme classificação original do TSE.",

    "qtd_2022":
        "Quantidade de registros de candidaturas da categoria em 2022.",

    "pct_2022":
        "Participação percentual da categoria sobre o total de candidaturas de 2022.",

    "qtd_2026":
        "Quantidade de registros de candidaturas da categoria em 2026.",

    "pct_2026":
        "Participação percentual da categoria sobre o total de candidaturas de 2026.",

    "variacao_qtd_pct":
        "Variação percentual da quantidade entre 2022 e 2026, calculada como "
        "(qtd_2026 - qtd_2022) / qtd_2022 * 100.",

    "variacao_pontos_percentuais":
        "Diferença entre pct_2026 e pct_2022, expressa em pontos percentuais."
}

aplicar_documentacao_tabela(
    TABELA_GOLD_GENERO,
    descricao_gold_genero,
    comentarios_gold_genero
)